In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import joblib
 
# Laden des Datensatzes
df = pd.read_csv('Datasets/apple_quality.csv')

# 2. Features (X) und Target (y) trennen
X = df.drop(['A_id','Quality'], axis=1).values # Löschen unnötige Spalten als Liste und Target
y = df['Quality'].values
# LabelEncoder sortiert die gefundenen Text-Kategorien standardmäßig alphabetisch
le = LabelEncoder()
y = le.fit_transform(y)

print("Shape of features:", X.shape)
print(X[0])
print(y[0])

### EDA Start

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
# Ignoriert nur Warnungen, die besagen, dass etwas in Zukunft wegfällt
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# Visualisierung der Klassenverteilung
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Quality')
plt.title('Verteilung der Apfel-Qualität')
plt.show()

In [ ]:
# Da der Datensatz viele Zeilen hat, nehmen wir ein Sample für den Plot
sns.pairplot(df.drop('A_id', axis=1), hue='Quality', corner=True)
plt.suptitle('Pairplot der Apfel-Merkmale', y=1.02)
plt.show()

In [ ]:
# Wir schmelzen das Dataframe für einen kombinierten Boxplot
df_melted = df.drop('A_id', axis=1).dropna().melt(id_vars='Quality')
df_melted['value'] = pd.to_numeric(df_melted['value'], errors='coerce')

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_melted, x='variable', y='value', hue='Quality')
plt.xticks(rotation=45)
plt.title('Merkmalsverteilung nach Qualität')
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
# Wir wandeln Quality kurz in Zahlen um, um sie in der Matrix zu sehen
df['Quality'] = df['Quality'].map({'good': 1, 'bad': 0})
# Acidity muss ggf. zu float konvertiert werden, falls es als String geladen wurde
df['Acidity'] = df['Acidity'].astype(float)
 
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Korrelationsmatrix der Merkmale')
plt.show()

### EDA Ende

In [ ]:
# Aufteilen des Datensatzes in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=0)
 
# Umwandeln der Daten in PyTorch-Tensoren
# Hinweis: from_numpy() teilt Speicher mit NumPy (keine Kopie) – effizienter als torch.tensor()
X_train = torch.from_numpy(X_train).float()
X_test  = torch.from_numpy(X_test).float()
y_train = torch.from_numpy(y_train).long()
y_test  = torch.from_numpy(y_test).long()
X_val   = torch.from_numpy(X_val).float()
y_val   = torch.from_numpy(y_val).long()
print(X_train.shape, y_train.shape)  # Ausgabe der Formen der Trainingsdaten
print(X_test.shape, X_val.shape)  # Ausgabe der Formen der Trainingsdaten

In [ ]:
# Definieren des neuronalen Netzes als Sequential-Modell
net = nn.Sequential(
    nn.Linear(7, 6),   # Eingabeschicht (7 Merkmale) -> Versteckte Schicht (6 Neuronen)
    nn.ReLU(),           # Aktivierungsfunktion: ReLU
    nn.Linear(6, 2)     # Versteckte Schicht (6 Neuronen) -> Ausgabeschicht (2 Klassen)
)
 
# Definieren des Verlustkriteriums und des Optimierers
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(net.parameters(), lr=0.01) # lr zw. 0.005 und 0.01
print(net)

In [ ]:
loss_history = []  # Liste zum Speichern der Verlustwerte

# Trainieren des neuronalen Netzes
net.train()  # Trainingsmodus aktivieren
for epoch in range(200):        # Epochen zw. 10 bei sehr großen Datensätzen und 1000
    optimizer.zero_grad()
    outputs = net(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item()) # Loss-Werte in der Liste abspeichern
 
    # Validierung nach jeder Epoche
    net.eval()
    with torch.no_grad():
        val_out  = net(X_val)
        val_loss = criterion(val_out, y_val)
    net.train()
 
    #print(f'Epoch {epoch:3d} | Loss: {loss.item():.4f}')
plt.plot(loss_history, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# Auswerten des neuronalen Netzes auf den Testdaten
net.eval()
with torch.no_grad():
    outputs = net(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = accuracy_score(y_test, predicted)
    print('Testgenauigkeit: ', accuracy)
 
# Abspeichern des trainierten Netzes
torch.save(net.state_dict(), 'apple_net.pth')
print("Trainiertes Netz abgespeichert als 'apple_net.pth'")
joblib.dump(le, 'label_encoder_apple.pkl')

In [ ]:
from sklearn.metrics import confusion_matrix

# 2. Confusion Matrix berechnen
# Wir wandeln die Tensoren zurück in numpy-Arrays um
cm = confusion_matrix(y_test.numpy(), predicted.numpy())

# 3. Visualisierung mit Seaborn
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le.classes_, 
            yticklabels=le.classes_)
plt.xlabel('Vorhergesagte Klasse')
plt.ylabel('Tatsächliche Klasse')
plt.title('Confusion Matrix: Apple Quality')
plt.show()

In [ ]:
# Wiederladen des State-Dicts
import torch
import torch.nn as nn

le = joblib.load('label_encoder_apple.pkl')

net = nn.Sequential(
    nn.Linear(7, 6),  # Eingabeschicht (4 Merkmale) -> Versteckte Schicht (10 Neuronen)
    nn.ReLU(),  # Aktivierungsfunktion: ReLU
    nn.Linear(6, 2)  # Versteckte Schicht (10 Neuronen) -> Ausgabeschicht (3 Klassen)
)
net.load_state_dict(torch.load('apple_net.pth', map_location=torch.device('cpu')))
for k, v in net.named_parameters():
    print(k,v)
net.eval()


In [ ]:
# Vorhersage mit dem trainierten Netz
# es wird eine Userabfrage gemacht, um die Merkmale einzugeben
# und eine Vorhersage zu treffen.

# Eingabe der vier Merkmale
features=[]
titel = ['size', 'weight', 'sweetness', 'crunchiness', 'juiciness', 'ripeness', 'acidity']
for titel in titel:
    val = float(input(f"{titel}: "))
    features.append(val)

# Erstellen eines Tensors aus den Eingabewerten
inputs = torch.tensor([features], dtype=torch.float32)

# Vorhersage treffen
with torch.no_grad():
    outputs = net(inputs)
    _, predicted = torch.max(outputs, 1)

# Ausgabe der Klassifizierungsaussage
print("Klasse: ", le.inverse_transform([predicted.item()])[0])

In [ ]:
# aus dem obigen Skript wird inputs übernommen, 
# die Vorhersagen werden als Wahrscheinlichkeiten ausgegeben

import torch.nn.functional as F

# Vorhersage des Netzes
with torch.no_grad():
    output = net(inputs)
    _, max_index = torch.max(output, 1)
    print("Vorhergesagte Klasse:", max_index.item())

# Klassenwahrscheinlichkeiten berechnen
probabilities = F.softmax(output, dim=1)
print("Klassenwahrscheinlichkeiten:", probabilities.numpy()[0])
